## Create NGrams with NLTK

In [1]:
import pandas as pd

In [2]:
import nltk

# from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import cess_esp
from nltk.tag import UnigramTagger, BigramTagger

In [3]:
nltk.download('cess_esp')
nltk.download('punkt')

[nltk_data] Downloading package cess_esp to /home/rca2t/nltk_data...
[nltk_data]   Package cess_esp is already up-to-date!
[nltk_data] Downloading package punkt to /home/rca2t/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
# Load the pre-trained Spanish POS tagger
cess_sents = cess_esp.tagged_sents()
unigram_tagger = UnigramTagger(cess_sents)
bigram_tagger = BigramTagger(cess_sents, backoff=unigram_tagger)

In [5]:
OHCO = "parte capit sent token".split()

In [6]:
TOKEN = pd.read_csv("recinos-TOKEN.csv").set_index(OHCO)

In [7]:
TOKEN = TOKEN.dropna(subset=['term_str'])

# Generate Ngram Tokens

In [8]:
widx = [f"w{i}" for i in range(4)]

In [9]:
pad_args = dict(pad_left=True, pad_right=True, left_pad_symbol='<s>', right_pad_symbol='</s>')

In [10]:
BG = TOKEN.groupby(OHCO[:3]).term_str.apply(lambda x: list(nltk.ngrams(x, 2, **pad_args))).apply(pd.Series).stack().apply(pd.Series)
TG = TOKEN.groupby(OHCO[:3]).term_str.apply(lambda x: list(nltk.ngrams(x, 3, **pad_args))).apply(pd.Series).stack().apply(pd.Series)
QG = TOKEN.groupby(OHCO[:3]).term_str.apply(lambda x: list(nltk.ngrams(x, 4, **pad_args))).apply(pd.Series).stack().apply(pd.Series)

In [11]:
BG.columns = widx[:2]
TG.columns = widx[:3]
QG.columns = widx[:4]

In [12]:
BG['ng_str'] = BG.apply(lambda x: ' '.join(x), axis=1).str.strip()
TG['ng_str'] = TG.apply(lambda x: ' '.join(x), axis=1).str.strip()
QG['ng_str'] = QG.apply(lambda x: ' '.join(x), axis=1).str.strip()

In [13]:
BG

w0         w1        ng_str
parte capit sent                                       
0     1     0    0         <s>       este      <s> este
                 1        este         es       este es
                 2          es         el         es el
                 3          el  principio  el principio
                 4   principio         de  principio de
...                        ...        ...           ...
4     12    72   10        que         se        que se
                 11         se      llama      se llama
                 12      llama      santa   llama santa
                 13      santa       cruz    santa cruz
                 14       cruz       </s>     cruz </s>

[37911 rows x 3 columns]

# Extract NG Types

In [14]:
BGT = BG[widx[:2]].value_counts().to_frame('n').sort_index()
TGT = TG[widx[:3]].value_counts().to_frame('n').sort_index()
QGT = QG[widx[:4]].value_counts().to_frame('n').sort_index()

In [15]:
BGT.index.names = widx[:2] #['w0', 'w1']
TGT.index.names = widx[:3] #['w0', 'w1', 'w2']
QGT.index.names = widx[:4] #['w0', 'w1', 'w2', 'w3']

In [18]:
BGT

n
w0       w1            
<s>      a           27
         abundancia   1
         acabaron     2
         acaso        8
         ademas       4
...                  ..
zotziha  la           1
zotzu    </s>         1
ztayul   asi          1
zuiva    de           1
zumbaban en           1

[19055 rows x 1 columns]